## បណ្តុះបណ្តាលម៉ូដែល CBoW

សៀវភៅកំណត់ត្រានេះជាផ្នែកមួយនៃ [ការបណ្តុះបណ្តាល AI សម្រាប់អ្នកចាប់ផ្តើម](http://aka.ms/ai-beginners)

ក្នុងឧទាហរណ៍នេះ យើងនឹងមើលការបណ្តុះបណ្តាលម៉ូដែលភាសា CBoW ដើម្បីទទួលបានកន្លែងបញ្ចូល Word2Vec ផ្ទាល់ខ្លួនរបស់យើង។ យើងនឹងប្រើទិន្នន័យ AG News ជាគម្របខ្ទង់អត្ថបទ។


In [ ]:
import torch
import torchtext
import os
import collections
import builtins
import random
import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ដំបូងតែងយកសំណុំទិន្នន័យរបស់យើង ហើយកំណត់ tokenizer និងវាក្យសព្ទ។ យើងនឹងកំណត់ `vocab_size` ទៅ ៥០០០ ដើម្បីកំណត់ការគណនាបន្តិច។


In [ ]:
def load_dataset(ngrams = 1, min_freq = 1, vocab_size = 5000 , lines_cnt = 500):
    tokenizer = torchtext.data.utils.get_tokenizer('basic_english')
    print("Loading dataset...")
    test_dataset, train_dataset  = torchtext.datasets.AG_NEWS(root='./data')
    train_dataset = list(train_dataset)
    test_dataset = list(test_dataset)
    classes = ['World', 'Sports', 'Business', 'Sci/Tech']
    print('Building vocab...')
    counter = collections.Counter()
    for i, (_, line) in enumerate(train_dataset):
        counter.update(torchtext.data.utils.ngrams_iterator(tokenizer(line),ngrams=ngrams))
        if i == lines_cnt:
            break
    vocab = torchtext.vocab.Vocab(collections.Counter(dict(counter.most_common(vocab_size))), min_freq=min_freq)
    return train_dataset, test_dataset, classes, vocab, tokenizer

In [ ]:
train_dataset, test_dataset, _, vocab, tokenizer = load_dataset()

Loading dataset...
Building vocab...


In [ ]:
def encode(x, vocabulary, tokenizer = tokenizer):
    return [vocabulary[s] for s in tokenizer(x)]

## ម៉ូដែល CBoW

CBoW រៀនទាយពាក្យមួយ dựa trênពាក្យជាប់ជាង $2N$ ។ ឧទាហរណ៍ បើ $N=1$ យើងនឹងទទួលបានគូដូចខាងក្រោមពីប្រយោគ *I like to train networks*: (like,I), (I, like), (to, like), (like,to), (train,to), (to, train), (networks, train), (train,networks)។ នៅទីនេះ ពាក្យដំបូងគឺពាក្យជាប់ដែលប្រើជាការបញ្ចូល ហើយពាក្យទីពីរជាពាក្យដែលយើងកំពុងទាយ។

ដើម្បីសង់បណ្តាញដើម្បីទាយពាក្យបន្ទាប់ យើងត្រូវផ្គត់ផ្គង់ពាក្យជាប់ជាការបញ្ចូល ហើយទទួលបានលេខពាក្យជាលទ្ធផល។ រចនាសម្ព័ន្ធនៃបណ្តាញ CBoW មានដូចខាងក្រោម៖

* ពាក្យបញ្ចូលត្រូវបានបញ្ចូនតាមស្រទាប់អំបិល។ ស្រទាប់អំបិលនេះជាស្រទាប់ embedding Word2Vec របស់យើង ដូច្នេះយើងនឹងកំណត់វាផ្ទាល់ជាអថេរ `embedder`។ យើងនឹងប្រើទំហំ embedding = 30 ក្នុងឧទាហរណ៍នេះ ទោះបីជាអ្នកអាចចង់សាកល្បងមិនលើសនេះ (word2vec ផ្ទាល់មួយមាន 300)
* វិទិក embedding នឹងត្រូវបញ្ចូនទៅស្រទាប់តែមួយដែលនឹងទាយពាក្យលទ្ធផល។ ដូច្នេះវាមានម៉េជ័រ `vocab_size`។

សម្រាប់លទ្ធផល ប្រសិនបើយើងប្រើ `CrossEntropyLoss` ជាអនុគមន៍ខាត ត្រូវផ្គត់ផ្គង់តែលេខពាក្យជាលទ្ធផលដែលរំពឹងទុក ដោយគ្មានការអ៊ិនកូដ one-hot។


In [ ]:
vocab_size = len(vocab)

embedder = torch.nn.Embedding(num_embeddings = vocab_size, embedding_dim = 30)
model = torch.nn.Sequential(
    embedder,
    torch.nn.Linear(in_features = 30, out_features = vocab_size),
)

print(model)

Sequential(
  (0): Embedding(5002, 30)
  (1): Linear(in_features=30, out_features=5002, bias=True)
)


## ការរៀបចំ​ទិន្នន័យ​បណ្តុះបណ្តាល

ឥឡូវនេះមកកត់កម្ម​មុខងារ​ផ្លូវការ​ដែល​នឹងគណនា​ជួរពាក្យ CBoW ពី​អត្ថបទ។ មុខងារនេះ​នឹង​អនុញ្ញាត​ឱ្យ​យើង​បញ្ជាក់​ទំហំ​បង្អួច និង​នឹង​ត្រឡប់​មក​ជូន​សំណុំ​ជួរ - ពាក្យ​បញ្ចូល និង​ពាក្យ​ផលបញ្ចេញ។ សូមចំណាំថា មុខងារនេះអាចប្រើប្រាស់លើពាក្យ ហើយក៏អាចប្រើលើវ៉ិចទ័រ/តិនស័រ ដែលនឹងអនុញ្ញាតឲ្យយើង encode អត្ថបទ មុននឹងផ្ញើវាទៅមុខងារ `to_cbow` ។


In [ ]:
def to_cbow(sent,window_size=2):
    res = []
    for i,x in enumerate(sent):
        for j in range(max(0,i-window_size),min(i+window_size+1,len(sent))):
            if i!=j:
                res.append([sent[j],x])
    return res

print(to_cbow(['I','like','to','train','networks']))
print(to_cbow(encode('I like to train networks', vocab)))

[['like', 'I'], ['to', 'I'], ['I', 'like'], ['to', 'like'], ['train', 'like'], ['I', 'to'], ['like', 'to'], ['train', 'to'], ['networks', 'to'], ['like', 'train'], ['to', 'train'], ['networks', 'train'], ['to', 'networks'], ['train', 'networks']]
[[232, 172], [5, 172], [172, 232], [5, 232], [0, 232], [172, 5], [232, 5], [0, 5], [1202, 5], [232, 0], [5, 0], [1202, 0], [5, 1202], [0, 1202]]


មកត្រៀមប្រមូលទិន្នន័យសម្រាប់បណ្តុះបណ្តាល។ យើងនឹងឆ្លងកាត់ព្រឹត្តិការណ៍ទាំងអស់ ហៅ `to_cbow` ដើម្បីទទួលបានបញ្ជីគូពាក្យ និងបន្ថែមគូទាំងនោះទៅ `X` និង `Y`។ ដើម្បីកាន់តែប្រសើរប្រើពេលវេលា យើងនឹងគិតត្រឹមតែព័ត៌មាន ១០,០០០ ឯកសារដំបូងប៉ុណ្ណោះ - អ្នកអាចលះបង់ការកំណត់នេះបានយ៉ាងងាយស្រួល ប្រសិនបើអ្នកមានពេលវេលាច្រើន ហើយចង់ទទួលបានការបញ្ចូលពាក្យល្អជាងនេះ :)


In [ ]:
X = []
Y = []
for i, x in zip(range(10000), train_dataset):
    for w1, w2 in to_cbow(encode(x[1], vocab), window_size = 5):
        X.append(w1)
        Y.append(w2)

X = torch.tensor(X)
Y = torch.tensor(Y)

យើងនឹងបម្លែងទិន្នន័យនោះទៅជាចំណុចទិន្នន័យមួយ ហើយបង្កើត dataloader:


In [ ]:
class SimpleIterableDataset(torch.utils.data.IterableDataset):
    def __init__(self, X, Y):
        super(SimpleIterableDataset).__init__()
        self.data = []
        for i in range(len(X)):
            self.data.append( (Y[i], X[i]) )
        random.shuffle(self.data)

    def __iter__(self):
        return iter(self.data)

យើងក៏នឹងបម្លែងទិន្នន័យនោះទៅជាឈុតទិន្នន័យមួយផងដែរ ហើយបង្កើត dataloader:


In [ ]:
ds = SimpleIterableDataset(X, Y)
dl = torch.utils.data.DataLoader(ds, batch_size = 256)

ឥឡូវនេះយើងនឹងធ្វើការបណ្តុះបណ្តាលពិតប្រាកដ។ យើងនឹងប្រើ `SGD` អុបទីម៉ាយហ្ស័រជាមួយអត្រាអានុបញ្ញាតិខ្ពស់។ អ្នកក៏អាចសាកល្បងលេងជាមួយអុបទីម៉ាយហ្ស័រផ្សេងទៀតដូចជា `Adam`។ យើង​នឹង​បណ្តុះ​បណ្តាលរយៈពេល ១០ epoch ជាចាប់ផ្តើម - ហើយ​អ្នក​អាច​រត់​កូដ​នេះ​ម្តងទៀត ប្រសិនបើ​ចង់​បាន ការបាត់បង់ (loss) ទាបជាងនេះ។


In [ ]:
def train_epoch(net, dataloader, lr = 0.01, optimizer = None, loss_fn = torch.nn.CrossEntropyLoss(), epochs = None, report_freq = 1):
    optimizer = optimizer or torch.optim.Adam(net.parameters(), lr = lr)
    loss_fn = loss_fn.to(device)
    net.train()

    for i in range(epochs):
        total_loss, j = 0, 0, 
        for labels, features in dataloader:
            optimizer.zero_grad()
            features, labels = features.to(device), labels.to(device)
            out = net(features)
            loss = loss_fn(out, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss
            j += 1
        if i % report_freq == 0:
            print(f"Epoch: {i+1}: loss={total_loss.item()/j}")

    return total_loss.item()/j

In [ ]:
train_epoch(net = model, dataloader = dl, optimizer = torch.optim.SGD(model.parameters(), lr = 0.1), loss_fn = torch.nn.CrossEntropyLoss(), epochs = 10)

Epoch: 1: loss=5.664632366860172
Epoch: 2: loss=5.632101973960962
Epoch: 3: loss=5.610399051405015
Epoch: 4: loss=5.594621561080262
Epoch: 5: loss=5.582538017415446
Epoch: 6: loss=5.572900234519603
Epoch: 7: loss=5.564951676341915
Epoch: 8: loss=5.558288112064614
Epoch: 9: loss=5.552576955031129
Epoch: 10: loss=5.547634165194347


5.547634165194347

## សាកល្បង Word2Vec

ដើម្បីប្រើប្រាស់ Word2Vec យើងចាប់ផ្តើមយកវ៉ិចទ័រដែលស្របគ្នានឹងពាក្យទាំងអស់ក្នុងវាក្យសម្ព័ន្ធរបស់យើង:


In [ ]:
vectors = torch.stack([embedder(torch.tensor(vocab[s])) for s in vocab.itos], 0)

មកមើលឧទាហរណ៍ មើលពីរបៀបដែលពាក្យ **Paris** ត្រូវបានរៀបចំជាទំព័រជួរៈ


In [ ]:
paris_vec = embedder(torch.tensor(vocab['paris']))
print(paris_vec)

tensor([-0.0915,  2.1224, -0.0281, -0.6819,  1.1219,  0.6458, -1.3704, -1.3314,
        -1.1437,  0.4496,  0.2301, -0.3515, -0.8485,  1.0481,  0.4386, -0.8949,
         0.5644,  1.0939, -2.5096,  3.2949, -0.2601, -0.8640,  0.1421, -0.0804,
        -0.5083, -1.0560,  0.9753, -0.5949, -1.6046,  0.5774],
       grad_fn=<EmbeddingBackward>)


វា​គឺជា​វិស្សមកាល​ក្នុង​ការ​ប្រើ Word2Vec ដើម្បី​ស្វែងរក​ពាក្យ​បញ្ញត្តិ​មួយស្មើ។ មុខងារ​ក្រោមនេះ​នឹង​បង្វិល​ទៅ​វិញ​នូវ​ពាក្យ​ដែល​ខ្ទង់ជិត `n` ទៅនឹង​ការ​បញ្ចូល​មួយ។ ដើម្បី​រក​ពួកវា​យើង​គណនា​គម្លាត​នៃ $|w_i - v|$ ដែល $v$ គឺជា​វ៉ិចទ័រ​តំណាងឱ្យ​ពាក្យ​បញ្ចូល​របស់​យើង ហើយ $w_i$ គឺជា​ការ​អ៊ិនកូដ​របស់​ពាក្យ​លេខ $i$ ក្នុង​ពាក្យសម្រង់។ យើង​ក្រោយមក​តម្រៀប​អារេ ហើយ​ដក​អនុសីត​គឺ `argsort` ហើយ​យក​ធាតុ `n` ដំបូង​នៃ​បញ្ជី ដែល​អ៊ិនកូដ​មាត្រានៃ​ទីតាំង​នៃ​ពាក្យ​ខ្ទង់​ជិត​ក្នុង​ពាក្យសម្រង់។


In [ ]:
def close_words(x, n = 5):
  vec = embedder(torch.tensor(vocab[x]))
  top5 = np.linalg.norm(vectors.detach().numpy() - vec.detach().numpy(), axis = 1).argsort()[:n]
  return [ vocab.itos[x] for x in top5 ]

close_words('microsoft')

['microsoft', 'quoted', 'lp', 'rate', 'top']

In [ ]:
close_words('basketball')

['basketball', 'lot', 'sinai', 'states', 'healthdaynews']

In [ ]:
close_words('funds')

['funds', 'travel', 'sydney', 'japan', 'business']

## លទ្ធផល

ការប្រើប្រាស់បច្ចេកវិទ្យាឆ្លាតវៃដូចជា CBoW គឺអាចបណ្តុះបណ្តាលម៉ូឌែល Word2Vec បាន។ អ្នកក៏អាចសាកល្បងបណ្តុះបណ្តាលម៉ូឌែល skip-gram ដែលត្រូវបានបណ្តុះបណ្តាលឱ្យទាយពាក្យជិតខាងមួយដែលផ្ដោតលើពាក្យកណ្តាល និងមើលថាវាធ្វើការល្អប៉ុណ្ណាដែរ។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការ​បញ្ជាក់​ថា​មិន​មាន​ភិក្ខុ**៖  
ឯកសារ​នេះ​ត្រូវ​បាន​បកប្រែ​ដោយ​ប្រើ​សេវាកម្ម​បកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator) ។ ក្នុង​អំឡុង​ខណៈពេល​យើង​ព្យាយាម​រក្សា​ភាសា​ឲ្យ​ត្រឹមត្រូវ សូម​ជ្រាបថា​ការ​បកប្រែ​ដោយ​ស្វ័យប្រវត្តិ​អាច​មាន​កំហុស ឬ​មិន​ត្រឹមត្រូវ​បាន។ ឯកសារ​ដើម​ក្នុង​ភាសា​មាតុភាសា​ត្រូវ​បាន​គេ​យក​ឲ្យ​ជា​លទ្ធផល​មាន​សុពលភាព។ សម្រាប់​ព័ត៌មាន​សំខាន់ៗ យើង​ស្នើ​ឲ្យ​បកប្រែ​ដោយ​មនុស្ស​ជំនាញ​វិជ្ជាជីវៈ។ យើង​មិន​ទទួល​ខុសត្រូវ​សម្រាប់​ការ​ច្របូកច្របល់ ឬ​ការ​បកប្រែ​មិន​ត្រឹមត្រូវ​ណាមួយ​ដែល​កើត​ឡើង​ពី​ការ​ប្រើប្រាស់​ការ​បកប្រែ​នេះ​នោះ​ទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
